# Granulocyte annotation

Takes the granulocyte object out of `immune_split.ipynb` and assigns
`cell_type` labels.

Two clustering passes: cluster and drop what does not belong, then re-cluster
what remains and annotate it. Removing cells changes every neighbourhood, so
the graph and UMAP are rebuilt in between.

Set `DATA` in `config.py` before running.

In [ ]:
import scanpy as sc

from config import DATA

SEED = 0

## 1. Load the granulocyte object

Written by `immune_split.ipynb`.

In [ ]:
adata = sc.read_h5ad(DATA / 'granulocytes.h5ad')
print(adata.shape)

## 2. First pass - cluster to find the junk

**The resolution below is an example, not a setting to copy.** Tune it by eye
until the cells you want to remove sit in a cluster of their own, and check the
result yourself.

In [ ]:
sc.pp.neighbors(adata, use_rep='X_scVI', n_neighbors=20, random_state=SEED)
sc.tl.umap(adata, random_state=SEED)

# example - tune by eye until the junk separates
sc.tl.leiden(adata, resolution=0.1, random_state=SEED)

print(f"{adata.obs['leiden'].nunique()} clusters")
sc.pl.umap(adata, color='leiden', palette='tab20', legend_loc='on data',
           frameon=False)

In [ ]:
sc.pl.umap(adata, color='n_genes_by_counts', frameon=False)

## 3. Drop the junk clusters

**The numbers below are an example.** They belong to the first pass at whatever
resolution was used above, so rebuild the list from the plots rather than
reusing it.

In [ ]:
JUNK = ['3', '4']

n_before = adata.n_obs
adata = adata[~adata.obs['leiden'].isin(JUNK)].copy()

print(f'{n_before:,} -> {adata.n_obs:,} cells ({n_before - adata.n_obs:,} dropped)')

## 4. Second pass - re-cluster what is left

In [ ]:
sc.pp.neighbors(adata, use_rep='X_scVI', n_neighbors=20, random_state=SEED)
sc.tl.umap(adata, random_state=SEED)

# example - tune by eye until the populations separate
sc.tl.leiden(adata, resolution=0.1, random_state=SEED)

print(f"{adata.obs['leiden'].nunique()} clusters")
sc.pl.umap(adata, color='leiden', palette='tab20', legend_loc='on data',
           frameon=False)

## 5. Markers

`Retnlg`, `Csf1` and `S100a9` separate the lineages. `Klf2`, `Cd164` and `Cd74`
are the genes named in the labels below - they do not distinguish neutrophils
from eosinophils, they distinguish each of those from its own sibling cluster.

In [ ]:
sc.pl.umap(adata, color=['Retnlg', 'Csf1', 'S100a9',
                        'Klf2', 'Cd164', 'Cd74'], frameon=False)

## 6. Assign cell types

**The dict below is an example.** The cluster numbers, and how many there are,
follow from whatever resolution was used above - so this mapping has to be
rebuilt rather than reused. Read the markers above and check every cluster
yourself.

The gene suffixes name what separates a cluster from its sibling rather than a
different lineage: `Neutrophils Klf2+` from plain `Neutrophils`, and
`Eosinophil Cd164+` from `Eosinophil Cd74+`.

In [ ]:
# example only - rebuild this for the clustering you get
CELL_TYPES = {
    '0': 'Neutrophils Klf2+',
    '1': 'Neutrophils',
    '2': 'Eosinophil Cd164+',
    '3': 'Eosinophil Cd74+',
    '4': 'Mast Cells',
}

adata.obs['cell_type'] = adata.obs['leiden'].map(CELL_TYPES).astype('category')

print(adata.obs['cell_type'].value_counts().to_string())

In [ ]:
sc.pl.umap(adata, color='cell_type', palette='tab20', frameon=False)

Composition per condition - the reason the annotation exists.

In [ ]:
sc.pl.umap(adata, color=['type', 'age'], frameon=False)

## 7. Save

In [ ]:
adata.write_h5ad(DATA / 'granulocytes_annotated.h5ad')
print(f'{adata.n_obs:,} cells, {adata.obs["cell_type"].nunique()} cell types')